# HW4 - Annie Sherwood

Exploratory analysis of the hotels dataset: loading and inspection,
hotel-type counts, missing values, continuous-variable statistics,
outlier treatment, mean imputation, and a kurtosis comparison.

**Note:** save `hotels.csv` in the same folder as this notebook before
running. Run the cells top to bottom (Kernel → Restart & Run All).

## Imports

In [ ]:
import glob
import os

import numpy as np
import pandas as pd

# Show every column/row of the summary tables instead of truncating them
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 120)

## Q1. Load & basic shape

Load the file with pandas, then print the column names, the data type of
every column, and the shape. The shape numbers come from `df.shape`, not
hard-coded values.

In [ ]:
EXPECTED_FILENAME = 'hotels.csv'

if os.path.exists(EXPECTED_FILENAME):
    csv_path = EXPECTED_FILENAME
else:
    # Fall back to any CSV with "hotel" in its name, in case the download
    # was renamed (e.g. "hotels (1).csv" or "Hotels.csv")
    matches = [path for path in glob.glob('*.csv')
               if 'hotel' in path.lower()]
    if not matches:
        raise FileNotFoundError(
            f"Could not find '{EXPECTED_FILENAME}' in {os.getcwd()!r}. "
            "Save the hotels CSV in the same folder as this notebook."
        )
    csv_path = matches[0]
    print(f"Note: using '{csv_path}'\n")

df = pd.read_csv(csv_path)

# 1) Full list of column names
print("Column names:")
print(df.columns.tolist())

# 2) Data type of every column
print("\nData types:")
print(df.dtypes)

# 3) Shape, taken directly from the loaded DataFrame
n_rows, n_cols = df.shape
print(f"\nObservations: {n_rows} Columns: {n_cols}")

## Q2. Hotel type counts

The hotel-type column is found from the data rather than assumed: it is
the text column whose name mentions "hotel" (or "type"), preferring the
one with the fewest distinct values, because a type column holds a few
categories.

In [ ]:
text_cols = df.select_dtypes(include='object').columns

# Candidate columns: text columns whose name mentions "hotel" or "type"
candidates = [col for col in text_cols
              if 'hotel' in col.lower() or 'type' in col.lower()]
if not candidates:
    # No helpful name, so fall back to all text columns
    candidates = list(text_cols)

# The column with the fewest distinct values is the category column
hotel_col = min(candidates, key=lambda col: df[col].nunique())
print(f"Hotel-type column: {hotel_col!r}")

In [ ]:
# Frequency table sorted by count, largest first
hotel_counts = (
    df[hotel_col]
    .value_counts()
    .sort_values(ascending=False)
    .rename_axis(hotel_col)
    .reset_index(name='count')
)
print(hotel_counts.to_string(index=False))

# Number of distinct hotel types, as a single integer
n_hotel_types = int(df[hotel_col].nunique())
print(n_hotel_types)

## Q3. Missing-value counts

Uses the `dataframe.method1().method2()` pattern: `isnull()` marks
every missing cell as `True`, and `sum()` adds up the `True` values in
each column.

In [ ]:
missing_counts = df.isnull().sum()
print(missing_counts)

## Q4. Continuous-variable summary table

A continuous variable is a numeric (int or float) column with **more
than 12** distinct non-missing values. (`nunique()` leaves out NaN by
default.) Kurtosis uses pandas' `kurt()`, which gives Fisher (excess)
kurtosis, so a normal distribution scores 0.

In [ ]:
DISTINCT_THRESHOLD = 12

numeric_cols = df.select_dtypes(include='number').columns
continuous_vars = [col for col in numeric_cols
                   if df[col].nunique(dropna=True) > DISTINCT_THRESHOLD]
print("Continuous variables:")
print(continuous_vars)

In [ ]:
# Mean, standard deviation, skewness and kurtosis per variable,
# in that column order
summary_table = pd.DataFrame({
    'mean': df[continuous_vars].mean(),
    'std': df[continuous_vars].std(),
    'skew': df[continuous_vars].skew(),
    'kurtosis': df[continuous_vars].kurt(),
})
summary_table.index.name = 'variable'
summary_table

## Q5. Outlier replacement

For each continuous variable, compute Q1, Q3 and IQR = Q3 - Q1 by hand.
A value is an outlier if it is below Q1 - 1.8 * IQR or above
Q3 + 1.8 * IQR. Outliers are replaced with `np.nan`. The table lists
only the variables that had at least one outlier.

In [ ]:
IQR_MULTIPLIER = 1.8

# Keep an untouched copy of the original data for later comparison
df_original = df.copy()

outlier_counts = {}
for col in continuous_vars:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_fence = q1 - IQR_MULTIPLIER * iqr
    upper_fence = q3 + IQR_MULTIPLIER * iqr

    # Missing values compare as False, so they are never counted
    is_outlier = (df[col] < lower_fence) | (df[col] > upper_fence)
    n_outliers = int(is_outlier.sum())

    if n_outliers > 0:
        # Replace outliers with missing values (float keeps NaN valid)
        df[col] = df[col].astype(float)
        df.loc[is_outlier, col] = np.nan
        outlier_counts[col] = n_outliers

outlier_table = pd.DataFrame({
    'variable': list(outlier_counts.keys()),
    'outliers_replaced': list(outlier_counts.values()),
})
print(outlier_table.to_string(index=False))

## Q6. Mean imputation

Only the variables changed in Q5 are imputed. Each one is filled with
the mean of the values left after outlier removal. `mean()` skips NaN,
so outliers are not part of that mean.

In [ ]:
treated_vars = list(outlier_counts.keys())

for col in treated_vars:
    col_mean = df[col].mean()
    df[col] = df[col].fillna(col_mean)

# Confirm the imputed variables have no missing values left
print("Missing values after imputation:")
print(df[treated_vars].isnull().sum())
print("\nAll treated variables complete:",
      bool(df[treated_vars].notnull().all().all()))

## Q7. Kurtosis comparison (before vs after)

Compare each treated variable's kurtosis from Q4 with its kurtosis after
outlier removal and mean imputation. Rows are sorted by the absolute
change, largest first.

In [ ]:
kurtosis_before = summary_table.loc[treated_vars, 'kurtosis']
kurtosis_after = df[treated_vars].kurt()

kurtosis_table = pd.DataFrame({
    'kurtosis_before': kurtosis_before,
    'kurtosis_after': kurtosis_after,
    'difference': kurtosis_after - kurtosis_before,
})
kurtosis_table.index.name = 'variable'

# Sort by the size of the change, ignoring its sign
kurtosis_table = kurtosis_table.reindex(
    kurtosis_table['difference'].abs()
    .sort_values(ascending=False).index
)
print(kurtosis_table)

In [ ]:
# State the largest change from the table above
top_var = kurtosis_table.index[0]
top_diff = kurtosis_table.loc[top_var, 'difference']
direction = 'less' if top_diff < 0 else 'more'
print(f"Largest kurtosis change: {top_var} "
      f"({kurtosis_table.loc[top_var, 'kurtosis_before']:.2f} -> "
      f"{kurtosis_table.loc[top_var, 'kurtosis_after']:.2f}, "
      f"difference {top_diff:.2f}), so its distribution became "
      f"{direction} heavy-tailed.")

**Conclusion:** The variable with the largest kurtosis change is the
first row of the table above (named in the printed sentence). Its
kurtosis dropped sharply, so its distribution became **less
heavy-tailed**. The extreme values that made up its long tail were
replaced with the mean, which piles more values near the centre.